# To be used with the cryoDRGN Anaconda Environment

Probably need to install tensorflow if you want to use GPU accelearted t-SNE calcs.

Worksheet to look at Latent space output from CRYODRGN.

This particular analysis is a combined VAE run with particles from GPR3:OA and GPR3:OEA



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from cryodrgn import utils

z = utils.load_pkl('Z-values/z.49.pkl')

# Convert the array to a DataFrame for easier manipulation with Pandas and Seaborn
data = pd.DataFrame(z, columns=[f'dim{i+1}' for i in range(z.shape[1])])

# Plotting using Seaborn
sns.set_theme(style="whitegrid")
sns.pairplot(data)  

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats 
from cryodrgn import utils
import umap

# Separate the first 65,000 entries
OA = z[:65000]
OEA = z[65000:]

# create to random subsets from each of the two datasets
np.random.seed(42)  # For reproducibility
indices_OA_30k_rand1 = np.random.choice(len(OA), size=30000, replace=False)
indices_OA_30k_rand2 = np.random.choice(len(OA), size=30000, replace=False)
indices_OEA_30k_rand1 = np.random.choice(len(OEA), size=30000, replace=False)
indices_OEA_30k_rand2 = np.random.choice(len(OEA), size=30000, replace=False)
OA_subset_A = OA[indices_OA_30k_rand1]
OA_subset_B = OA[indices_OA_30k_rand2]
OEA_subset_A = OEA[indices_OEA_30k_rand1]
OEA_subset_B = OEA[indices_OEA_30k_rand2]




# Save the separated data to new .pkl files 
#utils.save_pkl(OA, '128box/OA_Z.pkl')
#utils.save_pkl(OEA, '128box/OEA_Z.pkl')


# Calculate the magnitude of each vector in your dataset
OA_magnitudes = np.linalg.norm(OA, ord=2, axis=1)
OEA_magnitudes = np.linalg.norm(OEA, ord=2, axis=1)



# Plot histograms for both subsets
plt.figure(figsize=(20,10))

plt.subplot(1,2,1)
plt.hist(OA_magnitudes, bins=100, alpha=0.5, label='OA')
plt.hist(OEA_magnitudes, bins=100, alpha=0.5, label='OEA')
#plt.hist(OEA_magnitudes, bins=500,  alpha=0.5,label='OEA', color='orange')   
plt.title('Histogram of Vector Magnitudes')
plt.xlabel('Magnitude')
plt.xlim(0,6)
plt.ylim(0, 6000)
plt.ylabel('Frequency')
plt.legend()

# Boxplot for both subsets
plt.subplot(1,2,2)
plt.violinplot([OA_magnitudes, OEA_magnitudes], showmeans=True)
plt.title('ViolinPlot of Vector Magnitudes')
plt.xticks([1, 2], ['OA', 'OEA'])

plt.tight_layout()

plt.show()

# Perform t-test to compare means of both subsets
t_stat, p_value = stats.ttest_ind(OA_magnitudes, OEA_magnitudes)
print('T-test')
print('T-Statistic:', t_stat)
print('P-Value:', p_value)

u_stat, p_value = stats.mannwhitneyu(OA_magnitudes, OEA_magnitudes, alternative='two-sided')
print('Mann-Whitney U-Test')
print('U-Statistic:', u_stat)
print('P-Value:', p_value)

f_stat, p_value = stats.f_oneway(OA_magnitudes, OEA_magnitudes)
print('ANOVA Test')
print('F-Statistic:', f_stat)
print('P-Value:', p_value)

mean_OA = np.mean(OA_magnitudes)
mean_OEA = np.mean(OEA_magnitudes)
print('Mean OA Magnitude:', mean_OA, 'SD', np.std(OA_magnitudes))
print('Mean OEA Magnitude:', mean_OEA, 'SD', np.std(OEA_magnitudes))







In [ ]:
import umap
from pathlib import Path
import joblib

# Create a function to perform UMAP and save the results in a .pkl file
def perform_and_save_umap(data, filename):
    if filename.exists():
        print("Loading UMAP embeddings from disk...")
        data_umap = joblib.load(filename)
    else:
        print("Performing UMAP and saving the results to disk...")
        umap_reducer = umap.UMAP(n_components=2, metric='euclidean', min_dist=0.1, n_neighbors=50, spread=1.5, learning_rate=0.5, negative_sample_rate=10, init='pca', random_state=42)
        data_umap = umap_reducer.fit_transform(data)
        joblib.dump(data_umap, filename)
    return data_umap

# Define filenames for the UMAP embeddings
#filename_OA_subset_A = Path("umap_embeddings_OA_subset_A.pkl")
#filename_OA_subset_B = Path("umap_embeddings_OA_subset_B.pkl")
#filename_OEA_subset_A = Path("umap_embeddings_OEA_subset_A.pkl")
#filename_OEA_subset_B = Path("umap_embeddings_OEA_subset_B.pkl")
filename_OA = Path("umap_embeddings_OA.pkl")
filename_OEA = Path("umap_embeddings_OEA.pkl")

# Perform UMAP on subsets and save the results to disk
data_umap_OA_subset_A = perform_and_save_umap(OA_subset_A, filename_OA_subset_A)
data_umap_OA_subset_B = perform_and_save_umap(OA_subset_B, filename_OA_subset_B)
data_umap_OEA_subset_A = perform_and_save_umap(OEA_subset_A, filename_OEA_subset_A)
data_umap_OEA_subset_B = perform_and_save_umap(OEA_subset_B, filename_OEA_subset_B)
data_umap_OA = perform_and_save_umap(OA, filename_OA)
data_umap_OEA = perform_and_save_umap(OEA, filename_OEA)


In [ ]:
import tensorflow as tf
from sklearn.manifold import TSNE
import joblib
from pathlib import Path

# Create a function to perform tSNE and save the results in a .pkl file
def perform_and_save_tsne(data, filename, perplexity=100, n_iter=1000):
    if filename.exists():
        print("Loading t-SNE embeddings from disk...")
        data_tsne = joblib.load(filename)
    else:
        print("Performing t-SNE and saving the results to disk...")
        # Convert data from numpy array to tensor
        data_tensor = tf.convert_to_tensor(data, dtype=tf.float32)
        # Check if GPU is available
        if tf.test.is_gpu_available():
            print("Using GPU for tSNE")
            with tf.device('/GPU:0'):
                # Perform tSNE
                tsne = TSNE(perplexity=perplexity, n_iter=n_iter, random_state=42)
                data_tsne = tsne.fit_transform(data_tensor)
        else:
            print("Using CPU for tSNE")
            # Perform tSNE on CPU
            tsne = TSNE(perplexity=perplexity, n_iter=n_iter, random_state=42)
            data_tsne = tsne.fit_transform(data)
        joblib.dump(data_tsne, filename)
    return data_tsne

# Define filenames for the t-SNE embeddings
filename_OA_subset_A = Path("t-sne_embeddings_OA_subset_A.pkl")
filename_OA_subset_B = Path("t-sne_embeddings_OA_subset_B.pkl")
filename_OEA_subset_A = Path("t-sne_embeddings_OEA_subset_A.pkl")
filename_OEA_subset_B = Path("t-sne_embeddings_OEA_subset_B.pkl")
filename_OA = Path("t-sne_embeddings_OA.pkl")
filename_OEA = Path("t-sne_embeddings_OEA.pkl")


# Perform tSNE on subsets and save the results to disk
data_tsne_OA_subset_A = perform_and_save_tsne(OA_subset_A, filename_OA_subset_A)
data_tsne_OA_subset_B = perform_and_save_tsne(OA_subset_B, filename_OA_subset_B)
data_tsne_OEA_subset_A = perform_and_save_tsne(OEA_subset_A, filename_OEA_subset_A)
data_tsne_OEA_subset_B = perform_and_save_tsne(OEA_subset_B, filename_OEA_subset_B)
data_tsne_OA = perform_and_save_tsne(OA, filename_OA)
data_tsne_OEA = perform_and_save_tsne(OEA, filename_OEA)



In [ ]:
from scipy.stats import pearsonr
import numpy as np
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import euclidean_distances


# Do some stats on the tSNE data
# Calculated some distance metrics on each dataset
def calculate_distance_metrics(data):
    # Calculate pairwise distances
    pairwise_distances = euclidean_distances(data)
    
    # Calculate mean and standard deviation of distances
    mean_distance = np.mean(pairwise_distances)
    std_distance = np.std(pairwise_distances)
    
    return mean_distance, std_distance

# Calculate distance matrices
dist_matrix_OEA_subsetA = euclidean_distances(data_tsne_OEA_subset_A)
dist_matrix_OEA_subsetB = euclidean_distances(data_tsne_OEA_subset_B)
dist_matrix_OA_subsetA = euclidean_distances(data_tsne_OA_subset_A)
dist_matrix_OA_subsetB = euclidean_distances(data_tsne_OA_subset_B)
dist_matrix_OA = euclidean_distances(data_tsne_OA)
dist_matrix_OEA = euclidean_distances(data_tsne_OEA)


# Compare the matrices (e.g., using Pearson correlation)
similarity_score_OA, _ = pearsonr(dist_matrix_OA_subsetA.flatten(), dist_matrix_OA_subsetB.flatten())
print(f'Similarity Score for OA random sets: {similarity_score_OA}')
similarity_score_OEA, _ = pearsonr(dist_matrix_OEA_subsetA.flatten(), dist_matrix_OEA_subsetB.flatten())
print(f'Similarity Score for OEA random sets: {similarity_score_OEA}')
similarity_score_OA_OEA, _ = pearsonr(dist_matrix_OA.flatten(), dist_matrix_OEA.flatten())
print(f'Similarity Score for OA and OEA: {similarity_score_OA_OEA}')

In [ ]:
# Plot UMAP and tSNE Results for OA and OEA subsets
plt.figure(figsize=(20,10))
plt.subplot(2,2,1) 
plt.scatter(data_umap_OA_subset_A[:, 0], data_umap_OA_subset_A[:, 1], alpha=0.5, label='OA Subset A')
plt.scatter(data_umap_OA_subset_B[:, 0], data_umap_OA_subset_B[:, 1], alpha=0.5, label='OA Subset B')
plt.title('UMAP of OA Subsets')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.legend()
plt.subplot(2,2,2)
plt.scatter(data_umap_OEA_subset_A[:, 0], data_umap_OEA_subset_A[:, 1], alpha=0.5, label='OEA Subset A')
plt.scatter(data_umap_OEA_subset_B[:, 0], data_umap_OEA_subset_B[:, 1], alpha=0.5, label='OEA Subset B')
plt.title('UMAP of OEA Subsets')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.legend()
plt.subplot(2,2,3)
plt.scatter(data_tsne_OA_subset_A[:, 0], data_tsne_OA_subset_A[:, 1], alpha=0.5, label='OA Subset A')
plt.scatter(data_tsne_OA_subset_B[:, 0], data_tsne_OA_subset_B[:, 1], alpha=0.5, label='OA Subset B')
plt.title('tSNE of OA Subsets')
plt.xlabel('tSNE 1')
plt.ylabel('tSNE 2')
plt.legend()
plt.subplot(2,2,4)
plt.scatter(data_tsne_OEA_subset_A[:, 0], data_tsne_OEA_subset_A[:, 1], alpha=0.5, label='OEA Subset A')
plt.scatter(data_tsne_OEA_subset_B[:, 0], data_tsne_OEA_subset_B[:, 1], alpha=0.5, label='OEA Subset B')
plt.title('tSNE of OEA Subsets')
plt.xlabel('tSNE 1')
plt.ylabel('tSNE 2')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Perform PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=3)  # We are reducing to 3 dimensions for visualization
# Perform PCA on the original data
data_pca_OA = pca.fit_transform(OA)
data_pca_OEA = pca.fit_transform(OEA)

In [ ]:
# Plotting UMAP

UMAP_OA_magnitudes = np.linalg.norm(data_umap_OA, ord=2, axis=1)
UMAP_OEA_magnitudes = np.linalg.norm(data_umap_OEA, ord=2, axis=1)


plt.figure(figsize=(20,30))

plt.subplot(3,2,1)
plt.title('UMAP of latent space OA')   
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')
plt.scatter(data_umap_OA[:,0], data_umap_OA[:,1], alpha=0.9, label='OA',marker=".")

plt.subplot(3, 2, 2)  # 1 row, 2 columns, second plot
plt.hexbin(data_umap_OA[:, 0], data_umap_OA[:, 1], gridsize=50, cmap='Oranges', mincnt=1)
cbar = plt.colorbar()
cbar.set_label('Counts')
plt.title('UMAP Hexbin Visualization of the OA Dataset')
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')

plt.subplot(3,2,3)
plt.title('UMAP of latent space OEA')
plt.scatter(data_umap_OEA[:,0], data_umap_OEA[:,1], alpha=0.9, label='OEA',marker=".")
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')

plt.subplot(3,2,4)
plt.hexbin(data_umap_OEA[:, 0], data_umap_OEA[:, 1], gridsize=50, cmap='Oranges', mincnt=1)
cbar = plt.colorbar()
cbar.set_label('Counts')
plt.title('UMAP Hexbin Visualization of the OEA Dataset')
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')

plt.subplot(3,2,5)
plt.scatter(data_umap_OA[:,0], data_umap_OA[:,1], alpha=0.9, label='OA',marker=".")
plt.scatter(data_umap_OEA[:,0], data_umap_OEA[:,1], alpha=0.1, label='OEA',marker=".")
plt.title('UMAP of OA and OEA')
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')
plt.legend()

plt.subplot(3,2,6)
plt.hist(UMAP_OA_magnitudes, bins=100, alpha=0.5, label='OA')
plt.hist(UMAP_OEA_magnitudes, bins=100, alpha=0.5, label='OEA')
plt.title('Histogram of UMAP Vector Magnitudes')
plt.xlabel('Magnitude')
plt.ylabel('Frequency')
plt.legend()

plt.tight_layout()
plt.show()









In [ ]:
#PCA Plots 

plt.figure(figsize=(20, 20))

plt.subplot(2,2,1)
plt.scatter(data_pca_OA[:,0], data_pca_OA[:,1], alpha=0.9, label='OA',marker=".")
plt.scatter(data_pca_OEA[:,0], data_pca_OEA[:,1], alpha=0.3, label='OEA',marker=".")
plt.legend()
plt.title('PCA of 1st and 2nd Principal Component')
plt.xlabel('Principal Component 0')
plt.ylabel('Principal Component 1')


plt.subplot(2,2,2)
plt.scatter(data_pca_OA[:,0], data_pca_OA[:,2], alpha=0.9, label='OA',marker=".")
plt.scatter(data_pca_OEA[:,0], data_pca_OEA[:,2], alpha=0.3, label='OEA',marker=".")
plt.legend()
plt.title('PCA of 1st and 3rd Principal Component')
plt.xlabel('Principal Component 0')
plt.ylabel('Principal Component 2')

plt.subplot(2,2,3)
plt.scatter(data_pca_OA[:,1], data_pca_OA[:,2], alpha=0.9, label='OA',marker=".")
plt.scatter(data_pca_OEA[:,1], data_pca_OEA[:,2], alpha=0.3, label='OEA',marker=".")
plt.legend()
plt.title('PCA of 2nd and 3rd Principal Component')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')

plt.tight_layout()
plt.show()

In [ ]:
# t-SNE plots with skikit-learn, NOT VERY FAST
# Perform t-SNE on the original data

#from sklearn.manifold import TSNE

#tsne_OA = TSNE(n_components=2).fit_transform(OA)
#tsne_OEA = TSNE(n_components=2).fit_transform(OEA)


In [ ]:
# MultiPLOT tSNE 

plt.figure(figsize=(20, 30))  # Create a figure with size to accommodate two plots

# First subplot: Scatter plot for OA 
plt.subplot(3, 2, 1)  # 1 row, 2 columns, first plot
plt.scatter(data_tsne_OA[:, 0], data_tsne_OA[:, 1], alpha=0.3, label='OA', marker=".")
plt.legend()
plt.title('t-SNE Visualization of OA')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')

# Second subplot: Hexbin plot for the same data (you can add OEA if needed)
plt.subplot(3, 2, 2)  # 1 row, 2 columns, second plot
plt.hexbin(data_tsne_OA[:, 0], data_tsne_OA[:, 1], gridsize=40, cmap='Oranges', mincnt=1)
cbar = plt.colorbar()
cbar.set_label('Counts')
plt.title('tSNE Hexbin Visualization of the OA Dataset')
plt.xlabel('tSNE Component 1')
plt.ylabel('tSNE Component 2')

plt.subplot(3, 2, 3)  # 1 row, 2 columns, first plot
plt.scatter(data_tsne_OEA[:, 0], data_tsne_OEA[:, 1], alpha=0.3, label='OA', marker=".")
plt.legend()
plt.title('t-SNE Visualization of OEA')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')

plt.subplot(3, 2, 4)  # 1 row, 2 columns, second plot
plt.hexbin(data_tsne_OEA[:, 0], data_tsne_OEA[:, 1], gridsize=40, cmap='Oranges', mincnt=1)
cbar = plt.colorbar()
cbar.set_label('Counts')
plt.title('tSNE Hexbin Visualization of the OEA Dataset')
plt.xlabel('tSNE Component 1')
plt.ylabel('tSNE Component 2')

# Comparison Sublplot
plt.subplot(3, 2, 5)
plt.scatter(data_tsne_OA[:, 0], data_tsne_OA[:, 1], alpha=0.3, label='OA', marker=".")
plt.scatter(data_tsne_OEA[:, 0], data_tsne_OEA[:, 1], alpha=0.3, label='OEA', marker=".")
plt.legend()
plt.title('t-SNE Visualization of OA and OEA')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')

# Adjust layout for better spacing
plt.tight_layout()

# Show the figure
plt.show()

# Attempts to compare UMAP and tSNE

In [ ]:
import numpy as np
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

# Example data for OA_subset_A and OA_subset_B with 8 features each
#OA_subset_A = np.random.rand(100, 8)
#OA_subset_B = np.random.rand(100, 8)

# Apply t-SNE to the original data and generate corresponding coordinates (for visualization purposes)
#tsne = TSNE(n_components=2, random_state=42)
#data_tsne_OA_subset_A = tsne.fit_transform(OA_subset_A)
#data_tsne_OA_subset_B = tsne.fit_transform(OA_subset_B)

# Calculate cosine similarity between the original data and its t-SNE coordinates
cosine_similarity_matrix_OA_Subset = cosine_similarity(data_tsne_OA_subset_A, data_tsne_OA_subset_B)
cosine_similarity_matrix_OEA_Subset = cosine_similarity(data_tsne_OEA_subset_A, data_tsne_OEA_subset_B)
cosine_similarity_matrix_OA_OEA = cosine_similarity(data_tsne_OA, data_tsne_OEA)

# Create a scatter plot where points are colored based on their corresponding cosine similarity values
plt.figure(figsize=(20, 20))

plt.subplot(2, 2, 1)
sc1 = plt.scatter(data_tsne_OA_subset_A[:, 0], data_tsne_OA_subset_A[:, 1], c=cosine_similarity_matrix_OA_Subset[np.arange(len(cosine_similarity_matrix_OA_Subset)), np.arange(len(cosine_similarity_matrix_OA_Subset))], cmap='Oranges', s=0)
plt.colorbar(sc1, label='Cosine Similarity')
plt.title('tSNE Plot Colored by Cosine Similarity_OA_subset_A vs OA_subset_B')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')

plt.subplot(2, 2, 2)
sc2 = plt.scatter(data_tsne_OEA_subset_A[:, 0], data_tsne_OEA_subset_A[:, 1], c=cosine_similarity_matrix_OEA_Subset[np.arange(len(cosine_similarity_matrix_OEA_Subset)), np.arange(len(cosine_similarity_matrix_OEA_Subset))], cmap='Blues', s=50)
plt.colorbar(sc2, label='Cosine Similarity')
plt.title('tSNE Plot Colored by Cosine Similarity_OEA_subset_A vs OEA_subset_B')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')

plt.subplot(2,2,3)
sc3 = plt.scatter(data_tsne_OA[:, 0], data_tsne_OA[:, 1], c=cosine_similarity_matrix_OA_OEA[np.arange(len(cosine_similarity_matrix_OA_OEA)), np.arange(len(cosine_similarity_matrix_OA_OEA))], cmap='Greens', s=50)
plt.colorbar(sc3, label='Cosine Similarity')
plt.title('tSNE Plot Colored by Cosine Similarity_OA vs OEA')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')

plt.tight_layout()
plt.show()




In [ ]:

# Create hexbin plots where points are colored based on their corresponding cosine similarity values
plt.figure(figsize=(20, 15))

plt.subplot(2, 2, 1)
plt.hexbin(data_tsne_OA_subset_A[:, 0], data_tsne_OA_subset_A[:, 1], C=cosine_similarity_matrix_OA_Subset, cmap='coolwarm', gridsize=15)
plt.colorbar(label='Cosine Similarity')
plt.title('t-SNE Plot Colored by Cosine Similarity OA_subset_A vs OA_subset_B (Hexbin)')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')

plt.subplot(2, 2, 2)
plt.hexbin(data_tsne_OEA_subset_A[:, 0], data_tsne_OEA_subset_A[:, 1], C=cosine_similarity_matrix_OEA_Subset, cmap='coolwarm', gridsize=15)
plt.colorbar(label='Cosine Similarity')
plt.title('t-SNE Plot Colored by Cosine Similarity OEA_subset_A vs OEA_subset_B (Hexbin)')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')

plt.subplot(2, 2, 3)
plt.hexbin(data_tsne_OA[:, 0], data_tsne_OA[:, 1], C=cosine_similarity_matrix_OA_OEA, cmap='coolwarm', gridsize=15)
plt.colorbar(label='Cosine Similarity')
plt.title('t-SNE Plot Colored by Cosine Similarity OA vs OEA (Hexbin)')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')

plt.tight_layout()
plt.show()

# calculate the mean and standard deviation of the cosine similarity values

mean_cosine_similarity_OA_subset = np.mean(cosine_similarity_matrix_OA_Subset)
std_cosine_similarity_OA_subset = np.std(cosine_similarity_matrix_OA_Subset)
print(f'Mean Cosine Similarity for OA random sets: {mean_cosine_similarity_OA_subset}, SD: {std_cosine_similarity_OA_subset}')
mean_cosine_similarity_OEA_subset = np.mean(cosine_similarity_matrix_OEA_Subset)
std_cosine_similarity_OEA_subset = np.std(cosine_similarity_matrix_OEA_Subset)
print(f'Mean Cosine Similarity for OEA random sets: {mean_cosine_similarity_OEA_subset}, SD: {std_cosine_similarity_OEA_subset}')
mean_cosine_similarity_OA_OEA = np.mean(cosine_similarity_matrix_OA_OEA)
std_cosine_similarity_OA_OEA = np.std(cosine_similarity_matrix_OA_OEA)
print(f'Mean Cosine Similarity for OA and OEA: {mean_cosine_similarity_OA_OEA}, SD: {std_cosine_similarity_OA_OEA}')

# Only take the postive values for the cosine similarity and calculate the mean and standard deviation
cosine_similarity_matrix_OA_Subset_positive = cosine_similarity_matrix_OA_Subset[cosine_similarity_matrix_OA_Subset > 0]
mean_cosine_similarity_OA_subset_positive = np.mean(cosine_similarity_matrix_OA_Subset_positive)
std_cosine_similarity_OA_subset_positive = np.std(cosine_similarity_matrix_OA_Subset_positive)
print(f'Mean Cosine Similarity for OA random sets (positive values only): {mean_cosine_similarity_OA_subset_positive}, SD: {std_cosine_similarity_OA_subset_positive}')
cosine_similarity_matrix_OEA_Subset_positive = cosine_similarity_matrix_OEA_Subset[cosine_similarity_matrix_OEA_Subset > 0]
mean_cosine_similarity_OEA_subset_positive = np.mean(cosine_similarity_matrix_OEA_Subset_positive)
std_cosine_similarity_OEA_subset_positive = np.std(cosine_similarity_matrix_OEA_Subset_positive)
print(f'Mean Cosine Similarity for OEA random sets (positive values only): {mean_cosine_similarity_OEA_subset_positive}, SD: {std_cosine_similarity_OEA_subset_positive}')
cosine_similarity_matrix_OA_OEA_positive = cosine_similarity_matrix_OA_OEA[cosine_similarity_matrix_OA_OEA > 0]
mean_cosine_similarity_OA_OEA_positive = np.mean(cosine_similarity_matrix_OA_OEA_positive)
std_cosine_similarity_OA_OEA_positive = np.std(cosine_similarity_matrix_OA_OEA_positive)
print(f'Mean Cosine Similarity for OA and OEA (positive values only): {mean_cosine_similarity_OA_OEA_positive}, SD: {std_cosine_similarity_OA_OEA_positive}')

## Had a Play with PaCMAP dimension reduction..... 

# Didn't give data better than UMAP

In [ ]:

import pacmap
import numpy as np

# Assuming OA is a NumPy array or DataFrame with your data
# Perform PaCMAP
pacmap_reducer = pacmap.PaCMAP(n_components=2, n_neighbors=300, apply_pca=True)
data_pacmap_OA = pacmap_reducer.fit_transform(OA)
data_pacmap_OEA = pacmap_reducer.fit_transform(OEA)



In [ ]:
# MultiPLOT PaCMAP

plt.figure(figsize=(20, 30))  # Create a figure with size to accommodate two plots

# First subplot: Scatter plot for OA 
plt.subplot(3, 2, 1)  # 1 row, 2 columns, first plot
plt.scatter(data_pacmap_OA[:, 0], data_pacmap_OA[:, 1], alpha=0.3, label='OA', marker=".")
plt.legend()
plt.title('PaCMAP Visualization of OA')
plt.xlabel('PaCMAP Component 1')
plt.ylabel('PaCAMP Component 2')

# Second subplot: Hexbin plot for the same data (you can add OEA if needed)
plt.subplot(3, 2, 2)  # 1 row, 2 columns, second plot
plt.hexbin(data_pacmap_OA[:, 0], data_pacmap_OA[:, 1], gridsize=50, cmap='Oranges', mincnt=1)
cbar = plt.colorbar()
cbar.set_label('Counts')
plt.title('PaCMAP Hexbin Visualization of the OA Dataset')
plt.xlabel('PaCMAP Component 1')
plt.ylabel('PaCAMP Component 2')

plt.subplot(3, 2, 3)  # 1 row, 2 columns, first plot
plt.scatter(data_pacmap_OEA[:, 0], data_pacmap_OEA[:, 1], alpha=0.3, label='OA', marker=".")
plt.legend()
plt.title('PaCMAP Visualization of OEA')
plt.xlabel('PaCMAP Component 1')
plt.ylabel('PaCAMP Component 2')

plt.subplot(3, 2, 4)  # 1 row, 2 columns, second plot
plt.hexbin(data_pacmap_OEA[:, 0], data_pacmap_OEA[:, 1], gridsize=50, cmap='Oranges', mincnt=1)
cbar = plt.colorbar()
cbar.set_label('Counts')
plt.title('PaCMAP Hexbin Visualization of the OEA Dataset')
plt.xlabel('PaCMAP Component 1')
plt.ylabel('PaCAMP Component 2')

# Comparison Sublplot
plt.subplot(3, 2, 5)
plt.scatter(data_pacmap_OA[:, 0], data_pacmap_OA[:, 1], alpha=0.3, label='OA', marker=".")
plt.scatter(data_pacmap_OEA[:, 0], data_pacmap_OEA[:, 1], alpha=0.3, label='OEA', marker=".")
plt.legend()
plt.title('PaCMAP Visualization of OA and OEA')
plt.xlabel('PaCMAP Component 1')
plt.ylabel('PaCAMP Component 2')

# Adjust layout for better spacing
plt.tight_layout()

# Show the figure
plt.show()